# 05 - Other TWPAs

The previous tutorials used a three-wave-mixing kinetic-inductance TWPA (KI-TWPA). This notebook builds two other
designs with the same tools:

1. **A four-wave-mixing (4WM) KI-TWPA with a resonant notch and periodic modulation**, from M. Malnou, *Synthesis of
   artificial transmission lines tailored for traveling-wave parametric processes*, Phys. Rev. Applied **25**,
   054016 (2026). A resonant phase-matching (rpm) filter in one cell per supercell creates a narrow notch whose
   dispersion compensates the Kerr phase mismatch of the pump (the technique of K. O'Brien *et al.*, PRL **113**,
   157001 (2014) and C. Macklin *et al.*, Science **350**, 307 (2015)), while a periodic modulation of the shunt
   capacitors opens a stopband that suppresses the pump third harmonic. The rpm cell is a user-defined model, which
   also shows how to make a custom cell work with the `bloch` model.
2. **A Floquet-mode TWPA built from segments**: the nonlinearity is switched on and off smoothly along the line, so
   that the Floquet modes of the pumped line connect adiabatically to the linear modes at both ends and less signal is
   lost into frequency-conversion sidebands. The idea is from K. Peng *et al.*, *Floquet-Mode Traveling-Wave Parametric
   Amplifiers*, PRX Quantum **3**, 020306 (2022) (first experiment: arXiv:2503.11812, 2025). We apply it to the KI-TWPA
   of the previous tutorials, cut into segments with different nonlinearity.

In [ ]:
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np

from twpasolver import ModeSet, TWPAnalysis
from twpasolver.cmes.bloch import PORT_OUT
from twpasolver.cmes.solver_config import BlochSolverConfig
from twpasolver.logger import log
from twpasolver.matrices_arrays import ABCDArray
from twpasolver.models import TWPA, LCLfBaseCell
from twpasolver.twoport import TwoPortModel

log.setLevel("WARNING")

## 1. A 4WM KI-TWPA with a resonant notch and a modulated line

### The supercell

The line is a 50 $\Omega$ LC ladder of kinetic inductances $L_0 = 100$ pH, whose nonlinearity
$L = L_0\,[1 + (I/I_*)^2]$ with $I_* = 100\,\mu$A is twpasolver's TWPA model with no dc bias (pure 4WM). A supercell
holds 9 unit cells (Appendix I of the paper):

* eight cells with shunt capacitors $C_0 \dots C_4, C_4, C_3, C_2$ = 51.3, 48.7, 42.0, 34.3, 29.3, 29.3, 34.3,
  42.0 fF: two overlapped cosine modulations of the line impedance that open a stopband around 27 GHz;
* one rpm cell (shunt capacitor 48.7 fF) whose series inductor is shunted by a series LC branch
  ($L_r = 4.92$ nH, $C_r = 53.5$ fF).

The first eight are `LCLfBaseCell`s without finger inductance. The rpm cell is a user model: a `name`, a
`single_abcd` method and `bloch_nl_elements()`, which tells the `bloch` model that the nonlinear inductor $L_0$
carries the cell's output current. That is accurate away from the notch, where the LC branch is a high impedance;
near the pump it carries about 14 % more, in one cell out of nine, which we neglect.

In [ ]:
class RPMCell(TwoPortModel):
    """Shunt C, then a series inductor L in parallel with a series LC branch (rpm filter)."""

    name: Literal["RPMCell"] = "RPMCell"
    L: float
    C: float
    Lr: float
    Cr: float

    def single_abcd(self, freqs):
        w = 2 * np.pi * np.asarray(freqs, dtype=float)
        Z_branch = 1j * w * self.Lr + 1 / (1j * w * self.Cr)
        Zs = 1 / (1 / (1j * w * self.L) + 1 / Z_branch)
        Y = 1j * w * self.C
        abcd = np.empty((len(w), 2, 2), dtype=complex)
        abcd[:, 0, 0] = 1
        abcd[:, 0, 1] = Zs
        abcd[:, 1, 0] = Y
        abcd[:, 1, 1] = 1 + Y * Zs
        return ABCDArray(abcd)

    def bloch_nl_elements(self):
        return [(PORT_OUT, self.L)]


L0, Istar = 100e-12, 100e-6
caps = np.array([51.3, 48.7, 42.0, 34.3, 29.3, 29.3, 34.3, 42.0]) * 1e-15
C_rpm, Lr, Cr = 48.7e-15, 4.92e-9, 53.5e-15


def ktwpa(rpm: bool = True, modulated: bool = True, Ip0: float = 22e-6) -> TWPA:
    shunt = caps if modulated else np.full(8, np.mean([*caps, C_rpm]))
    last_C = C_rpm if modulated else shunt[0]
    cells = [LCLfBaseCell(L=L0, C=c, Lf=0.0) for c in shunt]
    cells.append(
        RPMCell(L=L0, C=last_C, Lr=Lr, Cr=Cr)
        if rpm
        else LCLfBaseCell(L=L0, C=last_C, Lf=0.0)
    )
    return TWPA(cells=cells, N=556, Istar=Istar, Idc=0.0, Ip0=Ip0)


an = TWPAnalysis(twpa=ktwpa(), freqs=(0.5, 45, 2e-3))
lin = an.linear
print(
    f"{an.twpa.N_tot} cells, Bloch impedance at 1 GHz: {lin.Zb[0, np.searchsorted(lin.freqs, 1.0)]:.1f} Ohm"
)
print("stopbands (GHz):", np.round(lin.stopbands, 2))
print(
    f"notch from (L0 + Lr) Cr: {1 / (2 * np.pi * np.sqrt((L0 + Lr) * Cr)) / 1e9:.2f} GHz"
)

### Linear response

The notch sits where the LC branch, capacitive below its own resonance, cancels the series inductor: an open in the
series branch at $1/(2\pi\sqrt{(L_0+L_r)C_r}) = 9.71$ GHz (the paper quotes 10 GHz). Just below it $k^*$ rises
steeply, which is the extra pump wavenumber the 4WM phase matching needs. The modulation stopband around 27 GHz
contains the pump third harmonic $3 f_p \approx 27.3$ GHz.

In [ ]:
fp, Ip0 = 9.1, 22e-6
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6, 6), sharex=True)
ax1.plot(lin.freqs, lin.S21_db)
ax2.plot(lin.freqs, lin.k_star)
for ax in (ax1, ax2):
    for lo, hi in lin.stopbands:
        ax.axvspan(lo, hi, color="gray", alpha=0.3)
    ax.axvline(fp, color="C3", ls=":")
    ax.axvline(3 * fp, color="C3", ls="--")
ax1.set_ylim(-20, 2)
ax1.set_ylabel("$|S_{21}|$ (dB)")
ax2.set_ylabel("$k^*$ (rad/cell)")
ax2.set_xlabel("Frequency (GHz)")
plt.show()

### Phase matching and gain

`phase_matching` with `mixing="4WM"` includes the self- and cross-phase modulation of the pump. Its optimal pump
lies below the notch (8.9 GHz here, from the linear mismatch averaged over 4-8 GHz). With the pump of the paper ($f_p = 9.1$ GHz, $I_p = 22\,\mu$A) both models give a flat
gain of about 18-19 dB across 4-7.5 GHz, close to the 20 dB from 4 to 8 GHz of the paper's harmonic-balance
simulation; the gain falls where the idler $2 f_p - f_s$ approaches the notch.

In [ ]:
pm = an.phase_matching(
    (4, 8, 0.1), np.arange(8.6, 9.6, 0.02), mixing="4WM", modes="basic_4wm"
)
print(f"optimal pump from the linear phase matching: {pm.optimal_pump_freq:.2f} GHz")
pm.plot()
plt.show()

signals = np.linspace(3.5, 8.5, 41)
fig, ax = plt.subplots(figsize=(6, 3.5))
for model in ("simple", "bloch"):
    r = an.gain(signals, pump=fp, Is0=1e-8, model=model, modes="basic_4wm")
    ax.plot(r.signal_freqs, r.gain_db, label=model)
no_rpm = TWPAnalysis(twpa=ktwpa(rpm=False), freqs=(0.5, 45, 2e-3))
r = no_rpm.gain(signals, pump=fp, Is0=1e-8, model="bloch", modes="basic_4wm")
ax.plot(r.signal_freqs, r.gain_db, label="bloch, without the rpm cell")
ax.set_xlabel("Signal frequency (GHz)")
ax.set_ylabel("Gain (dB)")
ax.legend()
plt.show()

Without the rpm cell the Kerr mismatch is left uncompensated and the gain collapses: the notch is what makes the
device an amplifier.

### Why the modulation: the pump third harmonic

In a KI-TWPA the line is nearly dispersionless far below its cutoff, so the pump third harmonic $3 f_p$ is almost
phase matched and grows along the line. We add it to the mode set and compare the design with a line whose shunt
capacitors all take their mean value (no modulation stopband). The harmonic lies inside the stopband of the design,
a regime that only the `bloch` model describes.

In [ ]:
P3 = ModeSet(["p", "s", "i", "p3"], [["i", "p+p-s"], ["p3", "p+p+p"]])
print(P3)
signals = np.linspace(4, 8, 17)
runs = {}
for name, twpa in (
    ("modulated (design)", ktwpa()),
    ("no modulation", ktwpa(modulated=False)),
):
    a = TWPAnalysis(twpa=twpa, freqs=(0.5, 45, 2e-3))
    r = a.gain(
        signals,
        pump=fp,
        Is0=1e-8,
        model="bloch",
        modes=P3,
        bloch_config=BlochSolverConfig(passes=60),
    )
    third = 20 * np.log10(
        np.abs(r.b_out[:, r.mode("p3")]) / np.abs(r.b_out[:, r.mode("p")])
    )
    runs[name] = r
    print(
        f"{name}: mean gain {np.mean(r.gain_db):.1f} dB, third harmonic / pump at the output "
        f"{np.mean(third):.1f} dB, converged {r.converged.sum()}/{len(signals)}"
    )
for name, r in runs.items():
    r.plot_currents(title=name)
plt.show()

The stopband suppresses the third harmonic at the output by more than 20 dB, in line with the 20-27 dB the paper
reports. In this four-mode model the unsuppressed harmonic also drains the pump, and the gain drops; the paper,
with many more mixing products, finds similar pump depletion with and without the stopband and attributes its
role to keeping the parametric process a clean 4WM one (a symmetric gain profile and a near-ideal quantum
efficiency). Larger mode sets place the conversion products $2 f_p + f_s$ near the edge of the 27 GHz stopband,
where the forward-backward iteration may fail to converge for some signals: such points are flagged in
`GainResult.converged` and should not be trusted.

## 2. A Floquet-mode TWPA from segments

In a uniformly pumped TWPA the signal does not only turn into idler photons: the pump also converts it to sidebands
(here $f_p \pm f_s$, $2 f_p \pm f_s$ and the corresponding idler terms). Photons left in those modes at the output
are lost information, which lowers the quantum efficiency. Peng *et al.* showed that switching the nonlinearity on
and off adiabatically along the line maps the input and output linear modes onto single Floquet modes of the pumped
line, so that the sideband content at the output nearly vanishes. The taper does not phase match anything; it cleans
the output.

We take the KI-TWPA of the previous tutorials, cut it into 40 segments, and scale its nonlinearity
($\propto 1/I_*^2$) with a Gaussian profile that keeps 10 % at the ends. The linear circuit is the same in every
segment, so the taper adds no reflections. Each segment holds a whole number of supercells, as segmented devices
require.

In [ ]:
base = TWPA.from_file("imskit_v3_holotype.json")
n_seg, floor, sigma = 40, 0.1, 0.2
u = (np.arange(n_seg) + 0.5) / n_seg
profile = floor + (1 - floor) * np.exp(-((u - 0.5) ** 2) / (2 * sigma**2))


def floquet_line(Ip0: float) -> TWPA:
    segments = [
        base.model_copy(
            deep=True,
            update={"N": base.N // n_seg, "Istar": base.Istar / np.sqrt(p), "Ip0": Ip0},
        )
        for p in profile
    ]
    return TWPA(cells=segments, Z0_ref=base.Z0_ref, Ip0=Ip0)


plt.figure(figsize=(5, 2.5))
plt.step(u, profile, where="mid")
plt.xlabel("Position along the line")
plt.ylabel("Relative nonlinearity")
plt.show()

The weaker average nonlinearity is compensated by a stronger pump (0.181 mA instead of 0.1 mA), which gives the
same mean gain. We keep the pump harmonics and two orders of frequency conversion, solve with the `bloch` model and
count, at the output, the photon flux $\propto |I|^2/f$ in the conversion modes relative to signal plus idler.

In [ ]:
EXT = ModeSet.extended_3wm(
    n_pump_harmonics=3, n_frequency_conversion=2, n_signal_harmonics=0
)
LOST = ("ps", "pi", "p2s", "p2i")
signals = np.linspace(4, 11, 15)
fp, cfg = 15.8, BlochSolverConfig(passes=60)
results = {}
for name, twpa in (
    ("uniform", base.model_copy(update={"Ip0": 1e-4})),
    ("Floquet", floquet_line(1.809e-4)),
):
    an = TWPAnalysis(twpa=twpa, freqs=(0.2, 80, 2e-3))
    r = an.gain(signals, pump=fp, Is0=1e-9, model="bloch", modes=EXT, bloch_config=cfg)
    f = an.problem(EXT, {"p": fp, "s": signals}).freqs
    flux = np.abs(r.b_out) ** 2 / np.abs(f)
    lost = sum(flux[:, r.mode(m)] for m in LOST)
    useful = flux[:, r.mode("s")] + flux[:, r.mode("i")]
    results[name] = (r, lost / useful)
    print(
        f"{name}: mean gain {np.mean(r.gain_db):.1f} dB, lost / (signal + idler) photons "
        f"{10 * np.log10(np.mean(lost / useful)):.1f} dB, converged {r.converged.sum()}/{len(signals)}"
    )

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
for name, (r, ratio) in results.items():
    ax1.plot(r.signal_freqs, r.gain_db, label=name)
    ax2.plot(r.signal_freqs, 10 * np.log10(ratio), label=name)
ax1.set_ylabel("Gain (dB)")
ax2.set_ylabel("Lost / useful photons (dB)")
for ax in (ax1, ax2):
    ax.set_xlabel("Signal frequency (GHz)")
    ax.legend()
plt.tight_layout()
plt.show()

At equal mean gain the tapered line leaves about 20 times (13 dB) fewer photons in the conversion modes. (One
signal point stops just above the convergence threshold in both runs; more passes give a cleaner curve.) The mode
currents along the line show why: in the uniform device the conversion modes keep growing up to the output, while
in the Floquet device they are populated in the middle and return to the signal and idler as the nonlinearity
switches off.

In [ ]:
for name, (r, _) in results.items():
    ax = r.plot_currents(title=f"{name}: mode currents, averaged over the signals")
plt.show()